# plant water-demand analysis

This Google Colab notebook:

1. discovers compatible SAPFLUXNET sites;
2. merges environmental, sap-flow, and plant metadata;
3. forecasts sap-flow demand one hour ahead;
4. compares the model with a persistence baseline; and
5. estimates a **data-derived soil-moisture adequacy threshold**.

> **Important:** SAPFLUXNET does not provide irrigation amounts. The threshold produced here is an empirical model response threshold—not a causal agronomic optimum or a recommended irrigation volume.

Run the cells from top to bottom. Runtime depends strongly on the number and size of sites.

##  Install and import packages

In [ ]:
# ============================================================
# 1. Package Installation
# Purpose: Install required scientific computing and ML dependencies
# ============================================================

%pip -q install pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
# ============================================================
# 2. Module Imports & Setup
# Purpose: Import core libraries for data handling, ML models, and metrics
# ============================================================

from pathlib import Path
import re
import warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import time

from IPython.display import display
import shutil

## Load the SAPFLUXNET CSV files folder


In [ ]:
# ============================================================
# 3. Environment & Directory Setup
# Purpose: Mount Google Drive and configure data/output directory paths
# ============================================================

from google.colab import drive
from pathlib import Path

# Connect Google Drive
drive.mount("/content/drive")

# Input and output folders
DATA_FOLDER = Path("/content/drive/MyDrive/0.1.3/0.1.3/csv/plant")
OUTPUT_FOLDER = Path("/content/sapfluxnet_results")

# Validate the data folder
if not DATA_FOLDER.exists():
    raise FileNotFoundError(f"Data folder was not found: {DATA_FOLDER}")

# Create the output folder
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Count all CSV files, including files inside subfolders
csv_count = len(list(DATA_FOLDER.rglob("*.csv")))

if csv_count == 0:
    raise ValueError(f"No CSV files were found inside: {DATA_FOLDER}")

print(f"Data folder: {DATA_FOLDER}")
print(f"CSV files found: {csv_count}")
print(f"Results folder: {OUTPUT_FOLDER}")

## Discover and select usable sites

In [ ]:
# ============================================================
# 4. Site Selection Threshold Parameters
# Purpose: Define filtering criteria (min rows and max missing SWC %)
# ============================================================

MIN_SITE_ROWS = 200
MAX_SOIL_MOISTURE_MISSING = 40.0

In [ ]:
# ============================================================
# 5. Site Discovery Function
# Purpose: Identify environmental, sap-flow, and metadata companion files
# ============================================================

def discover_sites(data_folder: Path) -> pd.DataFrame:
    """
    Explore all CSV files, identify environmental files by their columns,
    and find the corresponding sap-flow and plant metadata files.
    """

    records = []

    # Explore every CSV file, including files in subfolders
    csv_files = list(data_folder.rglob("*.csv"))

    print(f"Exploring {len(csv_files)} CSV files...")

    for csv_file in csv_files:
        try:
            # Read only column names first
            columns = pd.read_csv(
                csv_file,
                nrows=0
            ).columns.tolist()

            # Identify environmental files by column names
            swc_cols = [
                column
                for column in ["swc_shallow", "swc_deep"]
                if column in columns
            ]

            # Skip files without timestamp or soil moisture
            if "TIMESTAMP" not in columns or not swc_cols:
                continue

            # -------------------------------------------------
            # Extract site code
            # Previously handled by clean_site_code()
            # -------------------------------------------------

            stem = csv_file.stem

            # Remove Windows duplicate suffixes such as (1)
            stem = re.sub(r"\(\d+\)$", "", stem)

            # Remove the environmental file suffix
            site_code = stem.removesuffix("_env_data")

            folder = csv_file.parent

            # -------------------------------------------------
            # Find companion files
            # Previously handled by find_companion()
            # -------------------------------------------------

            sap_candidates = list(
                folder.glob(f"{site_code}_sapf_data*.csv")
            )

            plant_candidates = list(
                folder.glob(f"{site_code}_plant_md*.csv")
            )

            sap_file = (
                sorted(sap_candidates)[0]
                if sap_candidates
                else None
            )

            plant_file = (
                sorted(plant_candidates)[0]
                if plant_candidates
                else None
            )

            # Read timestamp and soil-moisture columns
            env = pd.read_csv(
                csv_file,
                usecols=["TIMESTAMP"] + swc_cols
            )

            # Calculate the percentage of rows where all
            # available soil-moisture columns are missing
            swc_missing = (
                env[swc_cols]
                .isna()
                .all(axis=1)
                .mean() * 100
            )

            # Determine whether the site can be used
            usable = bool(
                sap_file is not None
                and plant_file is not None
                and len(env) >= MIN_SITE_ROWS
                and swc_missing <= MAX_SOIL_MOISTURE_MISSING
            )

            records.append({
                "site_code": site_code,
                "env_file": str(csv_file),
                "sap_file": str(sap_file) if sap_file else None,
                "plant_file": str(plant_file) if plant_file else None,
                "rows": len(env),
                "start": env["TIMESTAMP"].min(),
                "end": env["TIMESTAMP"].max(),
                "swc_columns": ", ".join(swc_cols),
                "swc_missing_percent": round(swc_missing, 2),
                "usable": usable,
            })

        except Exception as exc:
            warnings.warn(
                f"Skipping {csv_file.name}: {exc}"
            )

    inventory = pd.DataFrame(records)

    print(f"Environmental datasets found: {len(inventory)}")

    if not inventory.empty:
        print(f"Usable datasets: {inventory['usable'].sum()}")

    return inventory

In [ ]:
# ============================================================
# 6. Execute Discovery & Generate Site Inventory
# Purpose: Scan data folder, build site inventory, and export CSV summary
# ============================================================

inventory = discover_sites(DATA_FOLDER)
inventory.to_csv(OUTPUT_FOLDER / "site_inventory.csv", index=False)

if inventory.empty:
    raise ValueError(
        "No *_env_data*.csv files with soil-moisture columns were found. "
        "Check the extracted folder structure."
    )

display(inventory.sort_values(["usable", "site_code"], ascending=[False, True]))
usable = inventory[inventory["usable"]].copy()
if usable.empty:
    raise ValueError(
        "No usable sites were found. A usable site needs env_data, sapf_data, "
        "plant_md, enough rows, and acceptable soil-moisture coverage."
    )
print("Usable sites:", ", ".join(usable["site_code"]))

# Extract Data

In [ ]:
# ============================================================
# 7. Merge Site Data & Plant Metadata
# Purpose: Load and merge environmental, sap-flow, and metadata per site
# ============================================================

site_dfs = {}

usable_inventory = inventory[inventory["usable"]].copy()

for _, row in usable_inventory.iterrows():
    site_code = row["site_code"]

    # Load the three files for this site
    env = pd.read_csv(
        row["env_file"],
        parse_dates=["TIMESTAMP"]
    )

    sap = pd.read_csv(
        row["sap_file"],
        parse_dates=["TIMESTAMP"]
    )

    plant = pd.read_csv(
        row["plant_file"]
    )

    # Ensure timestamps are valid
    env["TIMESTAMP"] = pd.to_datetime(
        env["TIMESTAMP"],
        errors="coerce",
        utc=True
    )

    sap["TIMESTAMP"] = pd.to_datetime(
        sap["TIMESTAMP"],
        errors="coerce",
        utc=True
    )

    env = env.dropna(subset=["TIMESTAMP"])
    sap = sap.dropna(subset=["TIMESTAMP"])

    # Environmental data may be hourly while sap flow may be
    # measured every 30 minutes. Convert both to hourly averages.
    env_hourly = (
        env
        .set_index("TIMESTAMP")
        .select_dtypes(include="number")
        .resample("1h")
        .mean()
        .reset_index()
    )

    sap_hourly = (
        sap
        .set_index("TIMESTAMP")
        .select_dtypes(include="number")
        .resample("1h")
        .mean()
        .reset_index()
    )

    # Convert sap-flow data from wide to long format
    sap_long = sap_hourly.melt(
        id_vars="TIMESTAMP",
        var_name="pl_code",
        value_name="sap_flow_cm3_h"
    )

    # Ensure plant codes have the same data type
    sap_long["pl_code"] = sap_long["pl_code"].astype(str)
    plant["pl_code"] = plant["pl_code"].astype(str)

    # Merge environmental and sap-flow data using time
    merged = env_hourly.merge(
        sap_long,
        on="TIMESTAMP",
        how="inner"
    )

    # Avoid duplicate plant rows
    plant_unique = plant.drop_duplicates(subset="pl_code")

    # Merge plant metadata using the plant code
    merged = merged.merge(
        plant_unique,
        on="pl_code",
        how="left"
    )

    # Add the site identifier
    merged["site_code"] = site_code

    # Store one merged DataFrame for this site
    site_dfs[site_code] = merged

    print(
        f"{site_code:<16} → "
        f"{merged.shape[0]:,} rows × "
        f"{merged.shape[1]} columns"
    )

print(f"\nCreated {len(site_dfs)} merged site DataFrames.")

In [ ]:
# ============================================================
# 8. Extract Individual Site DataFrames
# Purpose: Isolate sample site DataFrames for inspection
# ============================================================

#Split data into dataframes
idn_jam_oil_df = site_dfs["IDN_JAM_OIL"]
esp_pra_df = site_dfs["ESP_PRA"]
zaf_sou_sou_df = site_dfs["ZAF_SOU_SOU"]
gbr_dev_dro_df = site_dfs["GBR_DEV_DRO"]
arg_tre_df = site_dfs["ARG_TRE"]
arg_maz_df = site_dfs["ARG_MAZ"]
esp_san_b2_100_df = site_dfs["ESP_SAN_B2_100"]
gbr_dev_con_df = site_dfs["GBR_DEV_CON"]
fin_pet_df = site_dfs["FIN_PET"]
usa_mor_sf_df = site_dfs["USA_MOR_SF"]
esp_san_a2_45i_df = site_dfs["ESP_SAN_A2_45I"]

Explore data and clean

In [ ]:
# ============================================================
# 9. Inspect Data: IDN_JAM_OIL Site
# Purpose: Review summary statistics and structure for IDN_JAM_OIL
# ============================================================

idn_jam_oil_df.head()
idn_jam_oil_df.info()
idn_jam_oil_df.describe()

In [ ]:
# ============================================================
# 10. Inspect Data: ESP_PRA Site
# Purpose: Review summary statistics and structure for ESP_PRA
# ============================================================

esp_pra_df.head()
esp_pra_df.info()
esp_pra_df.describe()

In [ ]:
# ============================================================
# 11. Inspect Data: ZAF_SOU_SOU Site
# Purpose: Review summary statistics and structure for ZAF_SOU_SOU
# ============================================================

zaf_sou_sou_df.head()
zaf_sou_sou_df.info()
zaf_sou_sou_df.describe()

## Feature Selection
look like data has diffrent shapes and coulmns in for simplicity we will this focuses on:

Available soil water
Atmospheric drying pressure
Solar energy
Wind
Rainfall
Plant size and species



In [ ]:
# ============================================================
# 12. Pipeline Configuration Hyperparameters
# Purpose: Set forecast horizon, data splits, and quality thresholds
# ============================================================

FORECAST_HOURS = 1
MIN_SITE_ROWS = 200
MAX_SOIL_MOISTURE_MISSING = 40.0
TEST_FRACTION = 0.20
RANDOM_SEED = 42


In [ ]:
# ============================================================
# 13. Feature Selection Helper
# Purpose: Retain required environmental and plant physical characteristics
# ============================================================

def select_model_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Select the available environmental, plant and categorical features
    required for the model.
    """

    environmental_features = [
        "swc_shallow",
        "vpd",
        "sw_in",
        "ws",
        "precip",
    ]

    plant_features = [
        "pl_dbh",
        "pl_height",
        "pl_sapw_area",
    ]

    categorical_features = [
        "site_code",
        "pl_species",
    ]

    target_columns = [
        "TIMESTAMP",
        "pl_code",
        "sap_flow_cm3_h",
    ]

    requested_columns = (
        target_columns
        + environmental_features
        + plant_features
        + categorical_features
    )

    available_columns = [
        column
        for column in requested_columns
        if column in df.columns
    ]

    missing_columns = [
        column
        for column in requested_columns
        if column not in df.columns
    ]

    selected_df = df[available_columns].copy()

    print(f"Selected columns: {len(available_columns)}")

    if missing_columns:
        print("Missing columns:", missing_columns)

    return selected_df

In [ ]:
# ============================================================
# 14. Apply Feature Selection across Sites
# Purpose: Filter candidate features for each loaded site dataset
# ============================================================

selected_site_dfs = {}

for site_code, site_df in site_dfs.items():
    print(f"\nSite: {site_code}")

    selected_site_dfs[site_code] = select_model_features(
        site_df
    )

In [ ]:
# ============================================================
# 15. Site Filtering for Consistent Schema
# Purpose: Exclude sites lacking the full set of required feature columns
# ============================================================

#Columns that conatin only 13 columns will be chosen to be analyzed and traind model on
selected_site_dfs.pop('ESP_PRA', None)
selected_site_dfs.pop('USA_MOR_SF', None)

print("Remaining sites in selected_site_dfs:", selected_site_dfs.keys())

In [ ]:
# ============================================================
# 16. Analyze Missing Values Percentage
# Purpose: Compute percentage of nulls per feature across sites
# ============================================================

for site_code, df in selected_site_dfs.items():
    print(f"\n--- Missing values for site: {site_code} ---")
    missing_percentage = df.isnull().sum() * 100 / len(df)
    missing_percentage = missing_percentage[missing_percentage > 0].sort_values(ascending=False)
    if missing_percentage.empty:
        print("No missing values.")
    else:
        print(missing_percentage)


In [ ]:
# ============================================================
# 17. Analyze Missing Values Counts
# Purpose: Count absolute missing values per column across sites
# ============================================================

for site_code, df in selected_site_dfs.items():
    print(f"\n--- Missing values for site: {site_code} ---")
    missing_values= df.isnull().sum()
    missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
    if missing_values.empty:
        print("No missing values.")
    else:
        print(missing_values)


## Handling Missing Values and Data Preprocessing

In [ ]:
# ============================================================
# 18. Missing Value Imputation Routine
# Purpose: Interpolate short gaps and forward/backward fill numeric columns
# ============================================================

def handle_missing_values(
    df: pd.DataFrame,
    interpolation_limit: int = 3
) -> pd.DataFrame:
    """
    Handle missing values without imputing the prediction target.

    interpolation_limit=3 means that only short gaps of up to
    three consecutive hourly observations are interpolated.
    """

    df = df.copy()

    # Ensure correct timestamp format
    df["TIMESTAMP"] = pd.to_datetime(
        df["TIMESTAMP"],
        errors="coerce",
        utc=True
    )

    # Remove invalid timestamps
    df = df.dropna(subset=["TIMESTAMP"])

    # Sort each plant's time series
    df = df.sort_values(
        ["pl_code", "TIMESTAMP"]
    ).reset_index(drop=True)

    # ------------------------------------
    # 1. Time-varying environmental data
    # ------------------------------------

    time_features = [
        "vpd",
        "sw_in",
        "ws",
        "swc_shallow",
    ]

    available_time_features = [
        column
        for column in time_features
        if column in df.columns
    ]

    # Interpolate within each plant only.
    # limit_area="inside" prevents extrapolation at the edges.
    for column in available_time_features:
        df[column] = (
            df.groupby("pl_code")[column]
            .transform(
                lambda series: series.interpolate(
                    method="linear",
                    limit=interpolation_limit,
                    limit_area="inside"
                )
            )
        )

    # ------------------------------------
    # 2. Plant characteristics
    # ------------------------------------

    plant_features = [
        "pl_height",
        "pl_dbh",
        "pl_sapw_area",
    ]

    for column in plant_features:
        if column not in df.columns:
            continue

        # First: use the known value belonging to the same plant
        df[column] = (
            df.groupby("pl_code")[column]
            .transform(
                lambda series: series.fillna(
                    series.dropna().median()
                    if series.notna().any()
                    else np.nan
                )
            )
        )

        # Second: use the median for the same species
        if "pl_species" in df.columns:
            species_median = df.groupby(
                "pl_species"
            )[column].transform("median")

            df[column] = df[column].fillna(
                species_median
            )

        # Third: use the site's median
        df[column] = df[column].fillna(
            df[column].median()
        )

    return df

In [ ]:
# ============================================================
# 19. Clean & Impute Missing Data per Site
# Purpose: Execute missing value handling on selected site DataFrames
# ============================================================

cleaned_site_dfs = {}

for site_code, df in selected_site_dfs.items():
    cleaned_site_dfs[site_code] = handle_missing_values(
        df,
        interpolation_limit=3
    )

    print(
        f"{site_code}: "
        f"{df.isna().sum().sum():,} missing before → "
        f"{cleaned_site_dfs[site_code].isna().sum().sum():,} after"
    )

In [ ]:
# ============================================================
# 20. Data Preparation & Target Alignment Pipeline
# Purpose: Drop remaining target/SWC nulls and summarize cleaning results
# ============================================================

def prepare_model_data(
    site_dfs: dict
) -> tuple[dict, pd.DataFrame]:

    model_ready_dfs = {}  # Dictionary, not list
    cleaning_summary = []

    for site_code, df in site_dfs.items():
        cleaned = df.copy()
        original_rows = len(cleaned)

        # Remove pl_height because it is missing in several sites
        cleaned = cleaned.drop(
            columns=["pl_height"],
            errors="ignore"
        )

        missing_target = (
            cleaned["sap_flow_cm3_h"].isna().sum()
        )

        missing_swc = (
            cleaned["swc_shallow"].isna().sum()
        )

        # Remove rows missing the target or essential soil moisture
        cleaned = cleaned.dropna(
            subset=[
                "sap_flow_cm3_h",
                "swc_shallow",
            ]
        ).reset_index(drop=True)

        # Store one cleaned DataFrame for each site
        model_ready_dfs[site_code] = cleaned

        cleaning_summary.append({
            "site_code": site_code,
            "original_rows": original_rows,
            "missing_target": missing_target,
            "missing_swc": missing_swc,
            "removed_rows": original_rows - len(cleaned),
            "remaining_rows": len(cleaned),
            "remaining_missing_values": int(
                cleaned.isna().sum().sum()
            ),
        })

    summary_df = pd.DataFrame(cleaning_summary)

    return model_ready_dfs, summary_df

In [ ]:
# ============================================================
# 21. Execute Model Data Preparation
# Purpose: Generate model-ready site DataFrames and cleaning report
# ============================================================

model_ready_site_dfs, cleaning_summary_df = (
    prepare_model_data(cleaned_site_dfs)
)

display(cleaning_summary_df)

In [ ]:
# ============================================================
# 22. Post-Cleaning Null Check
# Purpose: Audit remaining null values in model-ready site datasets
# ============================================================

for site_code, df in model_ready_site_dfs.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\n--- {site_code} ---")

    if missing.empty:
        print("No missing values")
    else:
        print(missing)

In [ ]:
# ============================================================
# 23. Site-Specific Missing Value Cleanup
# Purpose: Drop remaining VPD nulls for ZAF_SOU_SOU dataset
# ============================================================

model_ready_site_dfs["ZAF_SOU_SOU"] = (
    model_ready_site_dfs["ZAF_SOU_SOU"]
    .dropna(subset=["vpd"])
    .reset_index(drop=True)
)

In [ ]:
# ============================================================
# 24. Final Dataset Missing Value Validation
# Purpose: Confirm zero missing values remain before concatenation
# ============================================================

total_missing = 0

for site_code, df in model_ready_site_dfs.items():
    missing_count = int(df.isna().sum().sum())
    total_missing += missing_count

    print(
        f"{site_code}: "
        f"{missing_count} missing values"
    )

print(f"\nTotal missing values: {total_missing}")

##Merge Data and split Training -Testing

In [ ]:
# ============================================================
# 25. Merge Sites into Combined Dataset
# Purpose: Concatenate model-ready data across all sites and inspect sample
# ============================================================

combined_df = pd.concat(
    model_ready_site_dfs.values(),
    ignore_index=True
)

combined_df["TIMESTAMP"] = pd.to_datetime(
    combined_df["TIMESTAMP"],
    errors="coerce",
    utc=True
)

combined_df = (
    combined_df
    .sort_values([
        "site_code",
        "pl_code",
        "TIMESTAMP"
    ])
    .reset_index(drop=True)
)

print("Combined dataset shape:", combined_df.shape)
print("Number of sites:", combined_df["site_code"].nunique())
print("Number of plants:", combined_df["pl_code"].nunique())
print("Missing values:", combined_df.isna().sum().sum())

In [ ]:
# ============================================================
# 26. Save Cleaned Dataset to Disk
# Purpose: Export combined preprocessed dataset to CSV
# ============================================================

#save the work into csv file
combined_df.to_csv(OUTPUT_FOLDER / "Processed_data.csv", index=False)

In [ ]:
# ============================================================
# 27. Exploratory Plot: Sap-Flow Distribution by Plant
# Purpose: Visualize sap-flow variation across individual plant specimens
# ============================================================

plt.figure(figsize=(15, 7))

# Get a list of unique plant codes
unique_plants = combined_df['pl_code'].unique()

# Select a few plants to plot (e.g., the first 3 or 4) to avoid over-plotting
# Adjust the number as needed for clarity
plants_to_plot = unique_plants[:4]

for plant_code in plants_to_plot:
    plant_df = combined_df[combined_df['pl_code'] == plant_code]
    plt.plot(plant_df['TIMESTAMP'], plant_df['sap_flow_cm3_h'], label=plant_code, alpha=0.7)

plt.title('Sap Flow over Time for Selected Plants from combined_df')
plt.xlabel('Timestamp')
plt.ylabel('Sap Flow (cm3/h)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "sap_flow_over_time.png", dpi=160)
plt.show()

### Data Distribution Histograms

In [ ]:
# ============================================================
# 28. Exploratory Plot: Feature Histograms
# Purpose: Plot distribution of all numeric features in the dataset
# ============================================================

numerical_cols = combined_df.select_dtypes(include=np.number).columns.tolist()

# Exclude 'sap_flow_cm3_h' as it was already plotted, and 'pl_code' and 'site_code' as they are categorical.
# Also exclude 'TIMESTAMP' as it's not suitable for histogram this way.
columns_to_exclude = ['sap_flow_cm3_h', 'pl_code', 'site_code', 'TIMESTAMP']
histogram_cols = [col for col in numerical_cols if col not in columns_to_exclude]

# Limit the number of histograms to avoid overwhelming output
max_histograms = 6 # You can adjust this number
selected_histogram_cols = histogram_cols[:max_histograms]

if selected_histogram_cols:
    plt.figure(figsize=(15, 10))
    for i, col in enumerate(selected_histogram_cols):
        plt.subplot(2, 3, i + 1) # Adjust subplot grid based on max_histograms
        sns.histplot(combined_df[col].dropna(), kde=True)
        plt.title(f'Distribution of {col}')
    plt.tight_layout()
    plt.savefig(OUTPUT_FOLDER / "numerical_data_histograms.png", dpi=160)
    plt.show()
else:
    print("No numerical columns available for histogram plotting.")

In [ ]:
# ============================================================
# 29. Dataset Statistical Summary
# Purpose: Output descriptive statistics for all combined variables
# ============================================================

display(combined_df.describe())

##  Create a chronological train/test split and train the model

In [ ]:
# ============================================================
# 30. Chronological Train-Test Split Function
# Purpose: Prevent data leakage via time-ordered train/test partitioning
# ============================================================

def chronological_split(
    data: pd.DataFrame,
    test_fraction: float = 0.20
):
    train_parts = []
    test_parts = []

    for site_code, site_df in data.groupby(
        "site_code"
    ):
        site_df = site_df.sort_values(
            "TIMESTAMP"
        )

        unique_times = np.sort(
            site_df["TIMESTAMP"].unique()
        )

        split_index = int(
            len(unique_times)
            * (1 - test_fraction)
        )

        split_time = unique_times[split_index]

        train_parts.append(
            site_df[
                site_df["TIMESTAMP"] < split_time
            ]
        )

        test_parts.append(
            site_df[
                site_df["TIMESTAMP"] >= split_time
            ]
        )

    train = pd.concat(
        train_parts,
        ignore_index=True
    )

    test = pd.concat(
        test_parts,
        ignore_index=True
    )

    return train, test

###Create target coulmn

The model predicts the plant’s sap flow at the next time step using current environmental conditions, plant characteristics, and previous sap-flow measurements.
sap_flow_cm3_h represents the volume of water transported through the plant’s sapwood per hour. It serves as an indicator of plant water use and transpiration demand. However, it does not directly represent the required irrigation amount.

When measurements are recorded hourly, the target represents sap flow one hour ahead.

In [ ]:
# ============================================================
# 31. Time-Lag Feature Engineering & Target Construction
# Purpose: Build lagged predictors and 1-hour ahead sap-flow target
# ============================================================

# Start from the combined, cleaned dataset
data = combined_df.copy()

data["TIMESTAMP"] = pd.to_datetime(
    data["TIMESTAMP"],
    errors="coerce",
    utc=True
)

data = (
    data
    .sort_values([
        "site_code",
        "pl_code",
        "TIMESTAMP"
    ])
    .reset_index(drop=True)
)

# Time features
hour = data["TIMESTAMP"].dt.hour
day = data["TIMESTAMP"].dt.dayofyear

data["hour_sin"] = np.sin(
    2 * np.pi * hour / 24
)

data["hour_cos"] = np.cos(
    2 * np.pi * hour / 24
)

data["doy_sin"] = np.sin(
    2 * np.pi * day / 365.25
)

data["doy_cos"] = np.cos(
    2 * np.pi * day / 365.25
)

# Separate the time series of every plant
grouped = data.groupby(
    ["site_code", "pl_code"],
    sort=False
)

# Previous sap-flow observations
data["sap_flow_lag_1h"] = (
    grouped["sap_flow_cm3_h"].shift(1)
)

data["sap_flow_lag_3h"] = (
    grouped["sap_flow_cm3_h"].shift(3)
)

data["sap_flow_roll_6h"] = (
    grouped["sap_flow_cm3_h"]
    .transform(
        lambda series:
        series.shift(1)
        .rolling(6, min_periods=3)
        .mean()
    )
)

# Previous soil-moisture observations
data["swc_shallow_lag_1h"] = (
    grouped["swc_shallow"].shift(1)
)

data["swc_shallow_roll_6h"] = (
    grouped["swc_shallow"]
    .transform(
        lambda series:
        series.shift(1)
        .rolling(6, min_periods=3)
        .mean()
    )
)

# Target: sap flow in the following observation
data["target"] = (
    grouped["sap_flow_cm3_h"].shift(-1)
)

# Replace infinite values
data = data.replace(
    [np.inf, -np.inf],
    np.nan
)

# Remove rows where engineered features cannot be calculated
required_columns = [
    "target",
    "sap_flow_lag_1h",
    "sap_flow_lag_3h",
    "sap_flow_roll_6h",
    "swc_shallow_lag_1h",
    "swc_shallow_roll_6h",
]

data = (
    data
    .dropna(subset=required_columns)
    .reset_index(drop=True)
)

print("Dataset shape:", data.shape)
print("Target created:", "target" in data.columns)
print("Remaining missing values:", data.isna().sum().sum())

In [ ]:
# ============================================================
# 32. Define Target Variable
# Purpose: Specify target column name for regression models
# ============================================================

TARGET_COLUMN = "target"

In [ ]:
# ============================================================
# 33. Feature Engineering Output Verification
# Purpose: Inspect sample rows of engineered lag and target variables
# ============================================================

print(data[[
    "TIMESTAMP",
    "site_code",
    "pl_code",
    "sap_flow_cm3_h",
    "sap_flow_lag_1h",
    "target",
]].head(10))

In [ ]:
# ============================================================
# 34. Feature Schema Identification & Categorical Profiling
# Purpose: Categorize numerical/categorical features and check unique values
# ============================================================

# Numeric feature candidates
numeric_candidates = [
    "swc_shallow",
    "vpd",
    "sw_in",
    "ws",
    "precip",
    "pl_dbh",
    "pl_sapw_area",
    "hour_sin",
    "hour_cos",
    "doy_sin",
    "doy_cos",
    "sap_flow_lag_1h",
    "sap_flow_lag_3h",
    "sap_flow_roll_6h",
    "swc_shallow_lag_1h",
    "swc_shallow_roll_6h",
]

# Categorical feature candidates
categorical_candidates = [
    "site_code",
    "pl_species",
]

# Keep only columns that actually exist
numeric = [
    column
    for column in numeric_candidates
    if column in data.columns
    and data[column].notna().any()
]

categorical = [
    column
    for column in categorical_candidates
    if column in data.columns
    and data[column].notna().any()
]

feature_columns = numeric + categorical

# Automatically identify the target name
if "target" in data.columns:
    TARGET_COLUMN = "target"
elif "target_sap_flow" in data.columns:
    TARGET_COLUMN = "target_sap_flow"
else:
    raise KeyError(
        "No target column was found. Run the feature-engineering cell first."
    )

print("Numeric features:")
print(numeric)

print("\nCategorical features:")
print(categorical)

print("\nTarget:", TARGET_COLUMN)
print("Total features:", len(feature_columns))

In [ ]:
# ============================================================
# 35. Execute Chronological Train-Test Split
# Purpose: Partition feature-engineered data into training and test sets
# ============================================================

TARGET_COLUMN = "target"
feature_columns = numeric + categorical

# Use the `data` DataFrame, which has all engineered features, for splitting
train, test = chronological_split(data, TEST_FRACTION)

print(f"Training rows: {len(train):,}")
print(f"Testing rows: {len(test):,}")
print(f"Training percentage: {len(train) / len(data) * 100:.2f}%")
print(f"Testing percentage: {len(test) / len(data) * 100:.2f}%")

## Model Training and Evaluation


In [ ]:
# ============================================================
# 36. Scikit-Learn Preprocessor Construction
# Purpose: Define StandardScaler and OneHotEncoder preprocessing pipeline
# ============================================================

def build_preprocessor(
    numeric_features,
    categorical_features
):
    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        ),
    ])

    return ColumnTransformer([
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        ),
    ])

In [ ]:
# ============================================================
# 37. Evaluation Metrics Helper
# Purpose: Calculate MAE, RMSE, and R2 regression performance scores
# ============================================================

def calculate_metrics(
    y_true,
    y_prediction
):
    return {
        "MAE": mean_absolute_error(
            y_true,
            y_prediction
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                y_true,
                y_prediction
            )
        ),
        "R2": r2_score(
            y_true,
            y_prediction
        ),
    }

In [ ]:
# ============================================================
# 38. HistGradientBoosting Model Pipeline
# Purpose: Configure HistGradientBoostingRegressor with feature preprocessor
# ============================================================

hist_model = Pipeline([
    (
        "preprocessor",
        build_preprocessor(
            numeric,
            categorical
        )
    ),
    (
        "model",
        HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=20,
            l2_regularization=1.0,
            random_state=42
        )
    ),
])

random_forest_model = Pipeline([
    (
        "preprocessor",
        build_preprocessor(
            numeric,
            categorical
        )
    ),
    (
        "model",
        RandomForestRegressor(
            n_estimators=200,
            max_depth=20,
            min_samples_leaf=5,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42
        )
    ),
])

In [ ]:
# ============================================================
# 39. Model Training & Evaluation Loop
# Purpose: Train HistGradientBoosting and RandomForest; record metrics and runtime
# ============================================================

models = {
    "HistGradientBoosting": hist_model,
    "Random Forest": random_forest_model,
}

evaluation_results = []
predictions = {}

for model_name, model in models.items():
    print(f"Training {model_name}...")

    start_time = time.time()

    model.fit(
        train[feature_columns],
        train[TARGET_COLUMN]
    )

    prediction = model.predict(
        test[feature_columns]
    )

    training_time = time.time() - start_time

    predictions[model_name] = prediction

    model_metrics = calculate_metrics(
        test[TARGET_COLUMN],
        prediction
    )

    evaluation_results.append({
        "model": model_name,
        **model_metrics,
        "training_time_seconds": training_time,
    })

    print(f"{model_name} complete.")

###Adding baseline

In [ ]:
# ============================================================
# 40. Baseline Model Evaluation
# Purpose: Evaluate 1-hour persistence baseline for performance benchmarking
# ============================================================

baseline_metrics = calculate_metrics(
    test[TARGET_COLUMN],
    test["sap_flow_lag_1h"]
)

evaluation_results.append({
    "model": "Persistence Baseline",
    **baseline_metrics,
    "training_time_seconds": 0,
})

In [ ]:
# ============================================================
# 41. Model Comparison Summary Table
# Purpose: Collate evaluation results across all models and sort by RMSE
# ============================================================

results_df = (
    pd.DataFrame(evaluation_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(
    results_df.style.format({
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "R2": "{:.4f}",
        "training_time_seconds": "{:.2f}",
    })
)

In [ ]:
# ============================================================
# 42. Performance Comparison Visualizations
# Purpose: Bar plots of MAE, RMSE, and R2 comparing models against baseline
# ============================================================

plt.figure(figsize=(15, 5))

# MAE Plot
plt.subplot(1, 3, 1)
sns.barplot(x='model', y='MAE', data=results_df, palette='viridis')
plt.title('Mean Absolute Error (MAE)')
plt.ylabel('MAE')
plt.xlabel('Model')
plt.xticks(rotation=45, ha='right')

# RMSE Plot
plt.subplot(1, 3, 2)
sns.barplot(x='model', y='RMSE', data=results_df, palette='viridis')
plt.title('Root Mean Squared Error (RMSE)')
plt.ylabel('RMSE')
plt.xlabel('Model')
plt.xticks(rotation=45, ha='right')

# R2 Plot
plt.subplot(1, 3, 3)
sns.barplot(x='model', y='R2', data=results_df, palette='viridis')
plt.title('R-squared (R2)')
plt.ylabel('R2 Score')
plt.xlabel('Model')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "model_performance_metrics.png", dpi=160)
plt.show()


##  Evaluate against a persistence baseline

In [ ]:
# ============================================================
# 43. Time-Series Forecasting Visualization
# Purpose: Plot actual vs. predicted sap flow across a 7-day test window
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 4)
)

metrics_to_plot = [
    ("MAE", "Lower is better"),
    ("RMSE", "Lower is better"),
    ("R2", "Higher is better"),
]

for axis, (metric, description) in zip(
    axes,
    metrics_to_plot
):
    sns.barplot(
        data=results_df,
        x="model",
        y=metric,
        hue="model",
        palette="viridis",
        legend=False,
        ax=axis
    )

    axis.set_title(
        f"{metric} — {description}"
    )
    axis.set_xlabel("")
    axis.tick_params(
        axis="x",
        rotation=25
    )

plt.suptitle(
    "Model Performance Comparison",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 44. Record Baseline Predictions
# Purpose: Append persistence baseline predictions to predictions dictionary
# ============================================================

predictions["Persistence Baseline"] = (
    test["sap_flow_cm3_h"].to_numpy()
)

In [ ]:
# ============================================================
# 45. Select Optimal Model
# Purpose: Identify the top-performing non-baseline regression model
# ============================================================

best_model_name = (
    results_df[
        results_df["model"]
        != "Persistence Baseline"
    ]
    .sort_values("RMSE")
    .iloc[0]["model"]
)

best_prediction = predictions[
    best_model_name
]

print("Best model:", best_model_name)

In [ ]:
# ============================================================
# 46. Prediction vs. Actual Scatter Plot
# Purpose: Plot actual vs. predicted scatter with ideal reference line
# ============================================================

plt.figure(figsize=(7, 6))

plt.scatter(
    test[TARGET_COLUMN],
    best_prediction,
    alpha=0.25,
    s=12
)

minimum_value = min(
    test[TARGET_COLUMN].min(),
    best_prediction.min()
)

maximum_value = max(
    test[TARGET_COLUMN].max(),
    best_prediction.max()
)

# Perfect-prediction reference line
plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    color="red",
    linestyle="--",
    label="Perfect prediction"
)

plt.xlabel("Observed sap flow")
plt.ylabel("Predicted sap flow")
plt.title(
    f"Observed vs Predicted — {best_model_name}"
)
plt.legend()
plt.tight_layout()
plt.show()

## Estimate empirical soil-moisture thresholds

In [ ]:
# ============================================================
# 47. Empirical Soil-Moisture Threshold Estimation
# Purpose: Sweep SWC levels per site to derive empirical adequacy thresholds
# ============================================================

# Must be before the loop
threshold_results = []
response_curves = {}

for site_code in train["site_code"].unique():

    print("Processing:", site_code)

    site_reference = train[
        train["site_code"] == site_code
    ].copy()

    if "sw_in" in site_reference.columns:
        daylight = site_reference[
            site_reference["sw_in"] > 50
        ].copy()

        if not daylight.empty:
            site_reference = daylight

    if (
        len(site_reference) < 100
        or site_reference["swc_shallow"].nunique() < 10
    ):
        print(
            f"Skipped {site_code}: "
            "not enough soil-moisture data."
        )
        continue

    lower_swc = site_reference[
        "swc_shallow"
    ].quantile(0.05)

    upper_swc = site_reference[
        "swc_shallow"
    ].quantile(0.95)

    swc_grid = np.linspace(
        lower_swc,
        upper_swc,
        60
    )

    reference_sample = site_reference.sample(
        n=min(500, len(site_reference)),
        random_state=42
    )

    mean_predictions = []

    for swc_value in swc_grid:

        scenario = reference_sample[
            feature_columns
        ].copy()

        scenario["swc_shallow"] = swc_value

        if "swc_shallow_lag_1h" in scenario.columns:
            scenario[
                "swc_shallow_lag_1h"
            ] = swc_value

        if "swc_shallow_roll_6h" in scenario.columns:
            scenario[
                "swc_shallow_roll_6h"
            ] = swc_value

        prediction = best_model.predict(
            scenario
        )

        mean_predictions.append(
            prediction.mean()
        )

    curve = pd.DataFrame({
        "swc_shallow": swc_grid,
        "predicted_sap_flow": mean_predictions,
    })

    curve["smoothed_prediction"] = (
        curve["predicted_sap_flow"]
        .rolling(
            window=5,
            center=True,
            min_periods=1
        )
        .mean()
    )

    minimum_prediction = (
        curve["smoothed_prediction"].min()
    )

    maximum_prediction = (
        curve["smoothed_prediction"].max()
    )

    target_level = (
        minimum_prediction
        + 0.90
        * (
            maximum_prediction
            - minimum_prediction
        )
    )

    response_correlation = curve[
        [
            "swc_shallow",
            "smoothed_prediction"
        ]
    ].corr(
        method="spearman"
    ).iloc[0, 1]

    if response_correlation > 0:

        threshold_candidates = curve[
            curve["smoothed_prediction"]
            >= target_level
        ]

        threshold = (
            threshold_candidates[
                "swc_shallow"
            ].iloc[0]
            if not threshold_candidates.empty
            else np.nan
        )

        threshold_reliable = (
            np.isfinite(threshold)
        )

    else:
        threshold = np.nan
        threshold_reliable = False

    response_curves[site_code] = curve

    # This must remain inside the site loop
    threshold_results.append({
        "site_code": site_code,
        "swc_5th_percentile": lower_swc,
        "swc_95th_percentile": upper_swc,
        "empirical_threshold": threshold,
        "target_response_level": target_level,
        "response_correlation": response_correlation,
        "threshold_reliable": threshold_reliable,
        "model": best_model_name,
        "interpretation": (
            "Reliable positive model response"
            if threshold_reliable
            else
            "No reliable positive response"
        ),
    })

# These lines must be outside the loop
thresholds_df = pd.DataFrame(
    threshold_results
)

print(
    f"\nCalculated thresholds for "
    f"{len(thresholds_df)} sites."
)

display(thresholds_df)

In [ ]:
# ============================================================
# 48. Soil Moisture Response Curves Plot
# Purpose: Visualize synthetic SWC response curves and detected thresholds
# ============================================================

number_of_sites = len(response_curves)

fig, axes = plt.subplots(
    number_of_sites,
    1,
    figsize=(9, 4 * number_of_sites)
)

if number_of_sites == 1:
    axes = [axes]

for axis, (site_code, curve) in zip(
    axes,
    response_curves.items()
):
    threshold = thresholds_df.loc[
        thresholds_df["site_code"] == site_code,
        "empirical_threshold"
    ].iloc[0]

    axis.plot(
        curve["swc_shallow"],
        curve["predicted_sap_flow"],
        color="lightblue",
        label="Raw model response"
    )

    axis.plot(
        curve["swc_shallow"],
        curve["smoothed_prediction"],
        color="blue",
        linewidth=2,
        label="Smoothed response"
    )

    if np.isfinite(threshold):
        axis.axvline(
            threshold,
            color="red",
            linestyle="--",
            label=f"90% threshold: {threshold:.3f}"
        )

    axis.set_title(
        f"Soil-moisture response — {site_code}"
    )
    axis.set_xlabel("Shallow soil-water content")
    axis.set_ylabel("Predicted sap flow")
    axis.legend()

plt.tight_layout()
plt.show()

Result Interpretation
Below the threshold, the model may indicate that insufficient soil moisture limits plant water use.
Above the threshold, further increases in soil moisture are associated with only limited improvement in sap flow.
A positive response_correlation indicates that predicted sap flow generally increases with soil moisture.
If the correlation is negative or the response curve is physiologically unreasonable, the estimated threshold should not be considered reliable for that site.

Thresholds should be estimated separately for each site. Differences in soil properties, plant species, and climatic conditions mean that swc_shallow values may not always be directly comparable across sites.

## Save and download all results

In [ ]:
# ============================================================
# 49. Export Models and Threshold Results
# Purpose: Persist best model artifact and threshold data to disk
# ============================================================

from pathlib import Path
from google.colab import files
import joblib
import shutil

OUTPUT_FOLDER = Path(
    "/content/sapfluxnet_best_model_results"
)

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

# Select the best ML model based on the lowest RMSE
best_result = (
    results_df[
        results_df["model"]
        != "Persistence Baseline"
    ]
    .sort_values("RMSE")
    .head(1)
)

best_model_name = best_result.iloc[0]["model"]
best_model = models[best_model_name]
best_prediction = predictions[best_model_name]

print("Best model:", best_model_name)
display(best_result)

In [ ]:
# ============================================================
# 50. Export Model Evaluation Metrics
# Purpose: Save best model performance metrics summary to CSV
# ============================================================

# Save the best model's evaluation metrics
best_result.to_csv(
    OUTPUT_FOLDER / "best_model_evaluation.csv",
    index=False
)

# Prepare the best model's test predictions
best_predictions_df = test[
    [
        "TIMESTAMP",
        "site_code",
        "pl_code",
        "pl_species",
        "sap_flow_cm3_h",
        TARGET_COLUMN,
    ]
].copy()

best_predictions_df = best_predictions_df.rename(
    columns={
        "sap_flow_cm3_h":
            "current_sap_flow",

        TARGET_COLUMN:
            "observed_future_sap_flow",
    }
)

best_predictions_df[
    "predicted_future_sap_flow"
] = best_prediction

best_predictions_df[
    "prediction_error"
] = (
    best_predictions_df[
        "observed_future_sap_flow"
    ]
    - best_predictions_df[
        "predicted_future_sap_flow"
    ]
)

best_predictions_df.to_csv(
    OUTPUT_FOLDER / "best_model_predictions.csv",
    index=False
)

In [ ]:
# ============================================================
# 51. Package Deliverables Archive
# Purpose: Create ZIP archive of results directory and trigger download
# ============================================================

archive = shutil.make_archive(
    str(OUTPUT_FOLDER.parent / OUTPUT_FOLDER.name),
    'zip',
    root_dir=OUTPUT_FOLDER.parent,
    base_dir=OUTPUT_FOLDER.name
)

print(f"Results saved in: {OUTPUT_FOLDER}")
print(f"ZIP file: {archive}")

files.download(archive)